# PRÁCTICA 2: SPARK 

In [89]:
import os

import pandas as pd
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark import SparkConf
from pyspark.sql import Row, SparkSession

In [90]:
conf = (
    SparkConf()
    .setAppName("Practica 2: Spark Adrian del Pozo")
    .set("spark.executor.memory", "7g")
    .set("spark.executor.cores", 5)
    .set("spark.dynamicAllocation.maxExecutors", 2)
)


In [91]:
spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()

In [92]:
print("=====================================================================================\n")
print("DataFrame y equema original (antes de transformaciones) para el DataFrame audiencias\n")
print("=====================================================================================\n")
audiencias = spark.read.load("/datos/practica_spark/muestra_audiencias.parquet").cache()
audiencias.show(10)
audiencias.printSchema()


DataFrame y equema original (antes de transformaciones) para el DataFrame audiencias




[Stage 1:======================================================>(136 + 1) / 137]

+------------+-------------------+-------------------+-------------------+-------------------+---------+
|id_contenido|   it_inicio_titulo|      it_fin_titulo|it_inicio_visionado|   it_fin_visionado|co_cadena|
+------------+-------------------+-------------------+-------------------+-------------------+---------+
|      4751.0|2016-06-29 14:46:00|2016-06-29 15:12:59|2016-06-29 14:46:50|2016-06-29 15:12:59|    117.0|
|     28817.0|2016-06-29 22:00:00|2016-06-29 22:23:25|2016-06-29 22:00:00|2016-06-29 22:13:37|     22.0|
|     63907.0|2016-06-29 20:55:00|2016-06-29 21:09:59|2016-06-29 21:04:34|2016-06-29 21:09:59|      2.0|
|       553.0|2016-06-29 21:35:00|2016-06-29 21:44:59|2016-06-29 21:35:00|2016-06-29 21:37:45|     73.0|
|     22418.0|2016-06-29 22:32:00|2016-06-30 00:48:59|2016-06-29 22:32:00|2016-06-29 23:27:07|     81.0|
|      3332.0|2016-06-29 09:00:00|2016-06-29 12:19:59|2016-06-29 10:32:41|2016-06-29 12:18:32|    129.0|
|     31530.0|2016-06-29 16:00:00|2016-06-29 17:45:58|2

In [93]:
print("===============================================================================\n")
print("DataFrame y equema original (antes de transformaciones) para el DataFrame info\n")
print("===============================================================================\n")
info = (spark.read.options(header=False, inferSchema = True, sep = ",").csv("/datos/practica_spark/info_contenidos.csv")).cache()
info.show(10)
info.printSchema()


DataFrame y equema original (antes de transformaciones) para el DataFrame info




+------+---+---+------+
|   _c0|_c1|_c2|   _c3|
+------+---+---+------+
|166702| SR| DR|EPISOD|
| 51017| SR| DR|EPISOD|
|149616| EN| MG|EPISOD|
|161669| DP| PR|EPISOD|
| 42048| DC| OT|EPISOD|
| 70503| DP| OT|EVENTO|
| 95420| CN| OE|TITULO|
| 66470| DP| PR|EPISOD|
|109122| IN| SR|EPISOD|
| 61130| DC| SD|TITULO|
+------+---+---+------+
only showing top 10 rows

root
 |-- _c0: integer (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)



#### 1.- Leer ambos ficheros a Spark DataFrame y hacer las transformaciones necesarias para conseguir los esquemas indicados.

In [94]:
# Ajustamos el esquema de audiencias
audiencias = (
    audiencias
    .withColumn("id_contenido", F.col("id_contenido").cast("integer"))
    .withColumn("co_cadena", F.col("co_cadena").cast("integer"))
    .withColumn("it_inicio_titulo", F.col("it_inicio_titulo").cast("string"))
    .withColumn("it_fin_titulo", F.col("it_fin_titulo").cast("string"))
    .withColumn("it_inicio_visionado", F.col("it_inicio_visionado").cast("string"))
    .withColumn("it_fin_visionado", F.col("it_fin_visionado").cast("string"))
)


In [95]:
# Reordenar las columnas para que coincidan con el orden del esquema solicitado
audiencias = audiencias.select(
    "id_contenido",
    "co_cadena",
    "it_inicio_titulo",
    "it_fin_titulo",
    "it_inicio_visionado",
    "it_fin_visionado"
)

In [96]:
print("===============================================================================\n")
print("DataFrame y equema tras las transformaciones para el DataFrame audiencias\n")
print("===============================================================================\n")
audiencias.show(10)
audiencias.printSchema()


DataFrame y equema tras las transformaciones para el DataFrame audiencias


+------------+---------+-------------------+-------------------+-------------------+-------------------+
|id_contenido|co_cadena|   it_inicio_titulo|      it_fin_titulo|it_inicio_visionado|   it_fin_visionado|
+------------+---------+-------------------+-------------------+-------------------+-------------------+
|        4751|      117|2016-06-29 14:46:00|2016-06-29 15:12:59|2016-06-29 14:46:50|2016-06-29 15:12:59|
|       28817|       22|2016-06-29 22:00:00|2016-06-29 22:23:25|2016-06-29 22:00:00|2016-06-29 22:13:37|
|       63907|        2|2016-06-29 20:55:00|2016-06-29 21:09:59|2016-06-29 21:04:34|2016-06-29 21:09:59|
|         553|       73|2016-06-29 21:35:00|2016-06-29 21:44:59|2016-06-29 21:35:00|2016-06-29 21:37:45|
|       22418|       81|2016-06-29 22:32:00|2016-06-30 00:48:59|2016-06-29 22:32:00|2016-06-29 23:27:07|
|        3332|      129|2016-06-29 09:00:00|2016-06-29 12:19:59|2016-06-29 10:32:41

In [97]:
# Renombramos las columnas del DataFrame de contenidos
info = (
    info
    .withColumnRenamed("_c0", "id_contenido")
    .withColumnRenamed("_c1", "genero")
    .withColumnRenamed("_c2", "subgenero")
    .withColumnRenamed("_c3", "tipo_contenido")
)
print("===============================================================================\n")
print("DataFrame y equema tras las transformaciones para el DataFrame info\n")
print("===============================================================================\n")
info.show(10)
info.printSchema()


DataFrame y equema tras las transformaciones para el DataFrame info


+------------+------+---------+--------------+
|id_contenido|genero|subgenero|tipo_contenido|
+------------+------+---------+--------------+
|      166702|    SR|       DR|        EPISOD|
|       51017|    SR|       DR|        EPISOD|
|      149616|    EN|       MG|        EPISOD|
|      161669|    DP|       PR|        EPISOD|
|       42048|    DC|       OT|        EPISOD|
|       70503|    DP|       OT|        EVENTO|
|       95420|    CN|       OE|        TITULO|
|       66470|    DP|       PR|        EPISOD|
|      109122|    IN|       SR|        EPISOD|
|       61130|    DC|       SD|        TITULO|
+------------+------+---------+--------------+
only showing top 10 rows

root
 |-- id_contenido: integer (nullable = true)
 |-- genero: string (nullable = true)
 |-- subgenero: string (nullable = true)
 |-- tipo_contenido: string (nullable = true)



#### 2.-¿Cuántos registros tiene cada DF? ¿Cuántos contenidos (id_contenido) únicos tiene cada uno? ¿Cuántas cadenas (co_cadena) hay en el primer DF?

In [98]:
#Número de registros de los DataFrames audiencias e info

num_audiencias = audiencias.count()
num_info = info.count()
print("===============================================================================\n")
print("¿Cuántos registros tiene cada DataFrame?\n")
print("===============================================================================\n")
print(f"El DataFrame 'audiencias' tiene {num_audiencias} registros.")
print(f"El DataFrame 'info' tiene {num_info} registros.")


¿Cuántos registros tiene cada DataFrame?


El DataFrame 'audiencias' tiene 17461469 registros.
El DataFrame 'info' tiene 178359 registros.


In [99]:
# Número de contenidos únicos en audiencias e info

num_id_contenido_audiencias = audiencias.select("id_contenido").distinct().count()
num_id_contenido_info = info.select("id_contenido").distinct().count()
print("===============================================================================\n")
print("¿Cuántos contenidos (id_contenido) únicos tiene cada uno?\n")
print("===============================================================================\n")
print(f"El DataFrame 'audiencias' tiene {num_id_contenido_audiencias} valores únicos en 'id_contenido'.")
print(f"El DataFrame 'info' tiene {num_id_contenido_info} valores únicos en 'id_contenido'.")


¿Cuántos contenidos (id_contenido) únicos tiene cada uno?


El DataFrame 'audiencias' tiene 159083 valores únicos en 'id_contenido'.
El DataFrame 'info' tiene 178359 valores únicos en 'id_contenido'.


In [100]:
#Número de cadenas en el DF info
print("===============================================================================\n")
print("¿Cuántas cadenas (co_cadena) hay en el primer DF?\n")
print("===============================================================================\n")
num_co_cadena = audiencias.select("co_cadena").distinct().count()
print(f"El DataFrame 'audiencias' tiene {num_co_cadena} cadenas únicas en 'co_cadena'.")


¿Cuántas cadenas (co_cadena) hay en el primer DF?




[Stage 27:==================================================>   (129 + 5) / 137]

El DataFrame 'audiencias' tiene 154 cadenas únicas en 'co_cadena'.


#### 3. En el primer dataset las columnas que marcan momentos (las variables que empiezan por it) están definidas como string, convertirlas a formato timestamp.

In [101]:
audiencias = (
    audiencias
    .withColumn("it_inicio_titulo", F.col("it_inicio_titulo").cast("timestamp"))
    .withColumn("it_fin_titulo", F.col("it_fin_titulo").cast("timestamp"))
    .withColumn("it_inicio_visionado", F.col("it_inicio_visionado").cast("timestamp"))
    .withColumn("it_fin_visionado", F.col("it_fin_visionado").cast("timestamp"))
)
print("===============================================================================\n")
print("Esquema del DataFrame audiencias con variables temporales en formato timestamp\n")
print("===============================================================================\n")
audiencias.printSchema()


Esquema del DataFrame audiencias con variables temporales en formato timestamp


root
 |-- id_contenido: integer (nullable = true)
 |-- co_cadena: integer (nullable = true)
 |-- it_inicio_titulo: timestamp (nullable = true)
 |-- it_fin_titulo: timestamp (nullable = true)
 |-- it_inicio_visionado: timestamp (nullable = true)
 |-- it_fin_visionado: timestamp (nullable = true)



#### 4.- Calcular ahora la duración en minutos de cada programa usando it_inicio_titulo y it_fin_titulo, una vez calculado descartar del dataset todos los registros donde la duración sea menor que un minuto.

In [102]:
# Calculamos la duración en minutos usando F.unix_timestamp
audiencias = audiencias.withColumn(
    "duracion_minutos",
    (F.unix_timestamp(F.col("it_fin_titulo")) - F.unix_timestamp(F.col("it_inicio_titulo"))) / 60
)
print("========================================================================================\n")
print("DataFrame y equema del DataFrame audiencias con la duracion en minutos de cada programa\n")
print("========================================================================================\n")
audiencias.show(10)
audiencias.printSchema()


DataFrame y equema del DataFrame audiencias con la duracion en minutos de cada programa


+------------+---------+-------------------+-------------------+-------------------+-------------------+------------------+
|id_contenido|co_cadena|   it_inicio_titulo|      it_fin_titulo|it_inicio_visionado|   it_fin_visionado|  duracion_minutos|
+------------+---------+-------------------+-------------------+-------------------+-------------------+------------------+
|        4751|      117|2016-06-29 14:46:00|2016-06-29 15:12:59|2016-06-29 14:46:50|2016-06-29 15:12:59|26.983333333333334|
|       28817|       22|2016-06-29 22:00:00|2016-06-29 22:23:25|2016-06-29 22:00:00|2016-06-29 22:13:37|23.416666666666668|
|       63907|        2|2016-06-29 20:55:00|2016-06-29 21:09:59|2016-06-29 21:04:34|2016-06-29 21:09:59|14.983333333333333|
|         553|       73|2016-06-29 21:35:00|2016-06-29 21:44:59|2016-06-29 21:35:00|2016-06-29 21:37:45| 9.983333333333333|
|       22418|       81|2016-06-29 22:32:

In [103]:
print("==================================================================================================\n")
print("Filtrado del DataFrame audiencias quedándonos con los registros con duración superior a un minuto\n")
print("==================================================================================================\n")
print(f"El DataFrame 'audiencias' tiene {audiencias.count()} registros antes del filtrado.")


Filtrado del DataFrame audiencias quedándonos con los registros con duración superior a un minuto




[Stage 34:==============================================>       (117 + 5) / 137]

El DataFrame 'audiencias' tiene 17461469 registros antes del filtrado.


In [104]:
# Descartar todos los registros donde la duración sea menor que un minuto.
audiencias = audiencias.filter(F.col("duracion_minutos") >= 1)
print(f"El DataFrame 'audiencias' tiene {audiencias.count()} registros después del filtrado.")
audiencias.show(10)
audiencias.printSchema()

El DataFrame 'audiencias' tiene 17446528 registros después del filtrado.
+------------+---------+-------------------+-------------------+-------------------+-------------------+------------------+
|id_contenido|co_cadena|   it_inicio_titulo|      it_fin_titulo|it_inicio_visionado|   it_fin_visionado|  duracion_minutos|
+------------+---------+-------------------+-------------------+-------------------+-------------------+------------------+
|        4751|      117|2016-06-29 14:46:00|2016-06-29 15:12:59|2016-06-29 14:46:50|2016-06-29 15:12:59|26.983333333333334|
|       28817|       22|2016-06-29 22:00:00|2016-06-29 22:23:25|2016-06-29 22:00:00|2016-06-29 22:13:37|23.416666666666668|
|       63907|        2|2016-06-29 20:55:00|2016-06-29 21:09:59|2016-06-29 21:04:34|2016-06-29 21:09:59|14.983333333333333|
|         553|       73|2016-06-29 21:35:00|2016-06-29 21:44:59|2016-06-29 21:35:00|2016-06-29 21:37:45| 9.983333333333333|
|       22418|       81|2016-06-29 22:32:00|2016-06-30 00:4

#### 5. ¿Cuál es la duración media de los contenidos?


In [105]:
print("==================================================================================================\n")
print("Duración media de los contenidos\n")
print("==================================================================================================\n")
# Calcular la duración media usando agg
duracion_media_I = audiencias.agg(
    F.avg(F.col("duracion_minutos")).alias("duracion_media (min)"))

# Mostrar el resultado
duracion_media_I.show()


Duración media de los contenidos




[Stage 41:====================================================> (133 + 4) / 137]

+--------------------+
|duracion_media (min)|
+--------------------+
|   78.69209916427764|
+--------------------+



In [106]:
duracion_media_II = audiencias.select(F.avg(F.col("duracion_minutos")).alias("duracion_media")).collect()[0]["duracion_media"]

print(f"La duración media de los contenidos es de {duracion_media_II:.2f} minutos.")

[Stage 44:====================================================> (132 + 5) / 137]

La duración media de los contenidos es de 78.69 minutos.


#### 6. Filtrar ahora todos los registros donde el inicio de visionado sea después del fin de visionado ya que se entienden que estos registros son erróneos.

In [107]:
# Filtramos los registros donde el inicio de visionado sea después del fin de visionado. Nos aseguramos tambien de quitar los registros con valores 
# nulos para it_inicio_visionado e it_fin_visionado, que podrian llevar a error. 
print("==================================================================================================\n")
print("Filtrado de los registros erróneos\n")
print("==================================================================================================\n")
print(f"El DataFrame 'audiencias' tiene {audiencias.count()} registros antes de filtrar los errores de visionado.")

audiencias = audiencias.filter(
    F.col("it_inicio_visionado").isNotNull() & 
    F.col("it_fin_visionado").isNotNull() & 
    (F.col("it_inicio_visionado") <= F.col("it_fin_visionado"))
)

print(f"El DataFrame 'audiencias' tiene {audiencias.count()} registros después de filtrar los errores de visionado.")
audiencias.show(10)
audiencias.printSchema()


Filtrado de los registros erróneos




El DataFrame 'audiencias' tiene 17446528 registros antes de filtrar los errores de visionado.


El DataFrame 'audiencias' tiene 17446521 registros después de filtrar los errores de visionado.
+------------+---------+-------------------+-------------------+-------------------+-------------------+------------------+
|id_contenido|co_cadena|   it_inicio_titulo|      it_fin_titulo|it_inicio_visionado|   it_fin_visionado|  duracion_minutos|
+------------+---------+-------------------+-------------------+-------------------+-------------------+------------------+
|        4751|      117|2016-06-29 14:46:00|2016-06-29 15:12:59|2016-06-29 14:46:50|2016-06-29 15:12:59|26.983333333333334|
|       28817|       22|2016-06-29 22:00:00|2016-06-29 22:23:25|2016-06-29 22:00:00|2016-06-29 22:13:37|23.416666666666668|
|       63907|        2|2016-06-29 20:55:00|2016-06-29 21:09:59|2016-06-29 21:04:34|2016-06-29 21:09:59|14.983333333333333|
|         553|       73|2016-06-29 21:35:00|2016-06-29 21:44:59|2016-06-29 21:35:00|2016-06-29 21:37:45| 9.983333333333333|
|       22418|       81|2016-06-29 2

#### 7. Calcular la duración de cada visualización en minutos.

In [108]:
# Calculamos la duración de cada visualización en minutos
print("==================================================================================================\n")
print("Cálculo de la duración de cada visualización en minutos\n")
print("==================================================================================================\n")
audiencias = audiencias.withColumn(
    "duracion_visionado_minutos",
    (F.unix_timestamp(F.col("it_fin_visionado")) - F.unix_timestamp(F.col("it_inicio_visionado"))) / 60
    
)

audiencias.show(10)


Cálculo de la duración de cada visualización en minutos


+------------+---------+-------------------+-------------------+-------------------+-------------------+------------------+--------------------------+
|id_contenido|co_cadena|   it_inicio_titulo|      it_fin_titulo|it_inicio_visionado|   it_fin_visionado|  duracion_minutos|duracion_visionado_minutos|
+------------+---------+-------------------+-------------------+-------------------+-------------------+------------------+--------------------------+
|        4751|      117|2016-06-29 14:46:00|2016-06-29 15:12:59|2016-06-29 14:46:50|2016-06-29 15:12:59|26.983333333333334|                     26.15|
|       28817|       22|2016-06-29 22:00:00|2016-06-29 22:23:25|2016-06-29 22:00:00|2016-06-29 22:13:37|23.416666666666668|        13.616666666666667|
|       63907|        2|2016-06-29 20:55:00|2016-06-29 21:09:59|2016-06-29 21:04:34|2016-06-29 21:09:59|14.983333333333333|         5.416666666666667|
|         553|       73|2016-06-29 

#### 8. Para cada contenido, cadena e inicio del título agregar (sumar) todos los minutos vistos.

In [109]:
print("==================================================================================================\n")
print("Para cada contenido, cadena e inicio del título agregar (sumar) todos los minutos vistos\n")
print("==================================================================================================\n")

# Sumamos minutos vistos por contenido, cadena e inicio del título

audiencias_agrupadas = audiencias.groupBy(
    F.col("id_contenido"),
    F.col("co_cadena"),
    F.col("it_inicio_titulo")
).agg(
    F.sum(F.col("duracion_visionado_minutos")).alias("total_minutos_vistos")
).orderBy(F.desc("total_minutos_vistos"))



audiencias_agrupadas.show(10)
audiencias_agrupadas.printSchema()


Para cada contenido, cadena e inicio del título agregar (sumar) todos los minutos vistos




[Stage 57:>                                                       (0 + 10) / 11]

+------------+---------+-------------------+--------------------+
|id_contenido|co_cadena|   it_inicio_titulo|total_minutos_vistos|
+------------+---------+-------------------+--------------------+
|      180487|       73|2016-06-21 21:00:00|   89340.26666666669|
|        9623|      129|2017-06-03 19:20:00|   88878.41666666669|
|       44828|      107|2017-05-10 20:45:00|   83510.43333333332|
|       13212|      129|2017-05-02 20:00:00|   81101.18333333335|
|      166655|       73|2016-06-27 18:00:00|   79502.33333333331|
|       37144|       73|2016-05-22 21:30:00|   78869.76666666665|
|       11707|      107|2017-03-08 20:45:00|   75884.98333333335|
|      144078|       73|2016-07-10 21:00:00|   72399.29999999999|
|      161287|       73|2016-06-17 21:00:00|   70393.23333333332|
|        3937|      129|2016-05-28 20:45:00|   70118.76666666666|
+------------+---------+-------------------+--------------------+
only showing top 10 rows

root
 |-- id_contenido: integer (nullable = true)


#### 9.- Guardar el nuevo DF con la información agregada en el esquema personal de hive con el nombre practica_audiencias_agregado (activar el modo overwrite).

In [110]:
tablas = spark.catalog.listTables("mbd31")

In [111]:
print("==================================================================================================\n")
print("Creación de la tabla en HIVE\n")
print("==================================================================================================\n")
pd.DataFrame(tablas)


Creación de la tabla en HIVE




,name,catalog,namespace,description,tableType,isTemporary
0,airbnb_clean,spark_catalog,[mbd31],None,EXTERNAL,False
1,aux_scoring,spark_catalog,[mbd31],None,MANAGED,False
2,aux_scoring_2,spark_catalog,[mbd31],None,MANAGED,False
3,aux_tweets,spark_catalog,[mbd31],None,MANAGED,False
4,aux_tweets_2,spark_catalog,[mbd31],None,MANAGED,False
5,clicks,spark_catalog,[mbd31],None,EXTERNAL,False
6,clicks_buscador,spark_catalog,[mbd31],None,EXTERNAL,False
7,clicks_classification_2,spark_catalog,[mbd31],None,EXTERNAL,False
8,clicks_fichas,spark_catalog,[mbd31],None,EXTERNAL,False
9,clicks_statistics,spark_catalog,[mbd31],None,EXTERNAL,False


In [112]:
audiencias_agrupadas.write.mode("overwrite").saveAsTable("mbd31.practica_audiencias_agregado")

In [113]:
print("==================================================================================================\n")
print("Comprobación de la creación de la tabla practica_audiencias_agregado en hive\n")
print("==================================================================================================\n")
tablas_II = spark.catalog.listTables("mbd31")
print(pd.DataFrame(tablas_II))


Comprobación de la creación de la tabla practica_audiencias_agregado en hive


                            name        catalog namespace description  \
0                   airbnb_clean  spark_catalog   [mbd31]        None   
1                    aux_scoring  spark_catalog   [mbd31]        None   
2                  aux_scoring_2  spark_catalog   [mbd31]        None   
3                     aux_tweets  spark_catalog   [mbd31]        None   
4                   aux_tweets_2  spark_catalog   [mbd31]        None   
5                         clicks  spark_catalog   [mbd31]        None   
6                clicks_buscador  spark_catalog   [mbd31]        None   
7        clicks_classification_2  spark_catalog   [mbd31]        None   
8                  clicks_fichas  spark_catalog   [mbd31]        None   
9              clicks_statistics  spark_catalog   [mbd31]        None   
10                   nobel_prize  spark_catalog   [mbd31]        None   
11                 nobel_prize_2  spark_cata

#### 10.- Usando el DF con información extra de los contenidos, conseguir/pegar en cada registro del DF agregado los campos: genero, subgenero y tipo_contenido usando para ello el campo en común id_contenido.

In [114]:
# Asignamos alias a los DataFrames
audiencias_agrupadas_alias = audiencias_agrupadas.alias("a")
info_alias = info.alias("i")

# Hacemos un join de los DataFrames usando alias
audiencias_completo = audiencias_agrupadas_alias.join(
    info_alias,
    on=F.col("a.id_contenido") == F.col("i.id_contenido"),
    how="left"
).select(
    F.col("a.id_contenido").alias("id_contenido"),
    F.col("a.co_cadena").alias("co_cadena"),
    F.col("a.it_inicio_titulo").alias("it_inicio_titulo"),
    F.col("a.total_minutos_vistos").alias("total_minutos_vistos"),
    F.col("i.genero").alias("genero"),
    F.col("i.subgenero").alias("subgenero"),
    F.col("i.tipo_contenido").alias("tipo_contenido")
)

print("================================================================================================================================================================\n")
print("Usando el DF con información extra de los contenidos, conseguir/pegar en cada registro del DF agregado los campos: genero, subgenero y tipo_contenido usando para ello el campo en común id_contenido.\n")
print("=================================================================================================================================================================\n")
audiencias_completo.show(10)
audiencias_completo.printSchema()


Usando el DF con información extra de los contenidos, conseguir/pegar en cada registro del DF agregado los campos: genero, subgenero y tipo_contenido usando para ello el campo en común id_contenido.




[Stage 88:====================================================> (133 + 4) / 137]

+------------+---------+-------------------+--------------------+------+---------+--------------+
|id_contenido|co_cadena|   it_inicio_titulo|total_minutos_vistos|genero|subgenero|tipo_contenido|
+------------+---------+-------------------+--------------------+------+---------+--------------+
|       76907|        2|2016-06-29 21:35:00|   379.0333333333334|    SR|       AN|        EPISOD|
|       44854|       25|2016-06-29 19:50:00|   89.88333333333334|    DC|       NZ|        EPISOD|
|      159165|       82|2016-06-29 22:45:00|             2277.35|    DC|       SD|        EPISOD|
|       21041|       32|2016-06-30 01:46:06|              403.75|    EN|       CN|        EPISOD|
|        4351|       39|2017-10-28 18:37:00|   265.6333333333333|    CN|       AC|        TITULO|
|       50394|       45|2017-10-28 23:55:00|  237.35000000000002|    SR|       PO|        EPISOD|
|        3017|      150|2017-11-25 10:53:00|  40.150000000000006|    IN|       DA|        EPISOD|
|        2770|      

#### 11.- Construir ahora las siguientes columnas sobre el nuevo DF:

- id_weekday: Día de la semana (en literal) del inicio del programa.
- id_month: Mes (en literal) del inicio del programa.
- id_inicio: Hora del inicio del programa.

In [115]:

# Agregamos las columnas solicitadas derivadas de it_inicio_titulo
audiencias_completo = audiencias_completo.withColumn(
    "id_weekday", F.date_format(F.col("it_inicio_titulo"), "EEEE")
).withColumn(
    "id_month", F.date_format(F.col("it_inicio_titulo"), "MMMM")
).withColumn(
    "id_inicio", F.date_format(F.col("it_inicio_titulo"), "HH")
)
print("=================================================================================================\n")
print("Construir ahora las siguientes columnas sobre el nuevo DF: id_weekday, id_month, id_inicio:\n")
print("=================================================================================================\n")

audiencias_completo.show()
audiencias_completo.printSchema()


Construir ahora las siguientes columnas sobre el nuevo DF: id_weekday, id_month, id_inicio:




[Stage 92:=====================================================>(136 + 1) / 137]

+------------+---------+-------------------+--------------------+------+---------+--------------+----------+--------+---------+
|id_contenido|co_cadena|   it_inicio_titulo|total_minutos_vistos|genero|subgenero|tipo_contenido|id_weekday|id_month|id_inicio|
+------------+---------+-------------------+--------------------+------+---------+--------------+----------+--------+---------+
|       76907|        2|2016-06-29 21:35:00|  379.03333333333336|    SR|       AN|        EPISOD| Wednesday|    June|       21|
|       44854|       25|2016-06-29 19:50:00|   89.88333333333334|    DC|       NZ|        EPISOD| Wednesday|    June|       19|
|      159165|       82|2016-06-29 22:45:00|  2277.3500000000004|    DC|       SD|        EPISOD| Wednesday|    June|       22|
|       21041|       32|2016-06-30 01:46:06|              403.75|    EN|       CN|        EPISOD|  Thursday|    June|       01|
|        4351|       39|2017-10-28 18:37:00|   265.6333333333333|    CN|       AC|        TITULO|  Satur

In [116]:
# Seleccionamos y renombramos las columnas para cumplir con el esquema solicitado
audiencias_final = audiencias_completo.select(
    F.col("id_contenido"),
    F.col("total_minutos_vistos").alias("minutos"),
    F.col("co_cadena"),
    F.col("genero"),
    F.col("subgenero"),
    F.col("tipo_contenido"),
    F.col("id_weekday"),
    F.col("id_month"),
    F.col("id_inicio")
)
print("=================================================================================================\n")
print("Seleccionar las columnas necesarias para conseguir el siguiente esquema ...\n")
print("=================================================================================================\n")

audiencias_final.show(10)
audiencias_final.printSchema()


Seleccionar las columnas necesarias para conseguir el siguiente esquema ...




[Stage 96:====================================================> (133 + 4) / 137]

+------------+------------------+---------+------+---------+--------------+----------+--------+---------+
|id_contenido|           minutos|co_cadena|genero|subgenero|tipo_contenido|id_weekday|id_month|id_inicio|
+------------+------------------+---------+------+---------+--------------+----------+--------+---------+
|         649|2008.1333333333337|       14|    OA|       NZ|         SERIE|    Monday|    July|       20|
|        5766|133.36666666666667|        7|    IN|       DA|         SERIE|    Monday|    July|       19|
|       17841|              67.6|       47|    OA|       CA|         SERIE|    Monday|    July|       16|
|       28536| 280.8666666666667|       96|    IN|       DA|        EPISOD|    Monday|    July|       14|
|      156909|2290.2333333333336|      131|    IF|       DE|        EPISOD|    Friday| January|       12|
|         429|2638.5500000000006|       73|    IF|       IF|         SERIE|    Friday| January|       21|
|       19464|213.45000000000002|       20|   

#### 12.- Con el DF generado hasta ahora, vamos a entrenar un modelo de Machine Learning, queremos explicar los minutos visualizados en función del resto de variables.

1.	StringIndexer: Convertimos las columnas categóricas en índices numéricos.

In [117]:


from pyspark.ml.feature import StringIndexer

# Creamos StringIndexer para cada columna categórica
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_index", handleInvalid="keep")
    for col in ["co_cadena", "genero", "subgenero", "tipo_contenido", "id_weekday", "id_month", "id_inicio"]
]

2.  OneHotEncoder: Convertimos los índices numéricos en vectores binarios.

In [118]:
from pyspark.ml.feature import OneHotEncoder

# Creamos OneHotEncoder para cada columna indexada
encoders = [
    OneHotEncoder(inputCol=f"{col}_index", outputCol=f"{col}_ohe", dropLast=True)
    for col in ["co_cadena", "genero", "subgenero", "tipo_contenido", "id_weekday", "id_month", "id_inicio"]
]

3. VectorAssembler: Juntamos todos los vectores binarios en una sola columna llamada features.

In [119]:
from pyspark.ml.feature import VectorAssembler

# Columnas que contienen los vectores OHE
ohe_columns = [
    "co_cadena_ohe",
    "genero_ohe",
    "subgenero_ohe",
    "tipo_contenido_ohe",
    "id_weekday_ohe",
    "id_month_ohe",
    "id_inicio_ohe"
]

# Creamos el VectorAssembler
vector_assembler = VectorAssembler(
    inputCols=ohe_columns,
    outputCol="features"
)

4. Pipeline: Agrupamos todas las etapas anteriores en una secuencia lógica.

In [120]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=indexers + encoders + [vector_assembler])

5. Dividimos  en 80% Train y 20% Test.

In [121]:
# Dividimos el DataFrame en Train (80%) y Test (20%) 
(trainDF, testDF) = audiencias_final.randomSplit([0.8, 0.2], seed=42)

# Cacheamos los DataFrames 
trainDF.cache()
testDF.cache()

DataFrame[id_contenido: int, minutos: double, co_cadena: int, genero: string, subgenero: string, tipo_contenido: string, id_weekday: string, id_month: string, id_inicio: string]

In [122]:
n = audiencias_final.count()
print("===========================================================================================================\n")
print("MODELO DE MACHINE LEARNING PARA LA ESTIMACIÓN DE LOS MINUTOS VISUALIZADOS EN FUNCIÓN DEL RESTO DE VARIABLES\n")
print("===========================================================================================================\n")
print("Total de registros en el DF original: {}".format(n))
print("Porcentaje en  train respecto al original: {}".format(1.0 * trainDF.count() / n))
print("Porcentaje en  test respecto al original: {}".format(1.0 * testDF.count() / n))


MODELO DE MACHINE LEARNING PARA LA ESTIMACIÓN DE LOS MINUTOS VISUALIZADOS EN FUNCIÓN DEL RESTO DE VARIABLES


Total de registros en el DF original: 1478208


Porcentaje en  train respecto al original: 0.800030171667316


[Stage 121:=================================>                  (129 + 10) / 200]

Porcentaje en  test respecto al original: 0.1999698283326839


In [123]:
p_model = pipeline.fit(trainDF)

In [124]:
p_model.transform(trainDF).limit(5).toPandas()

,id_contenido,minutos,co_cadena,genero,subgenero,tipo_contenido,id_weekday,id_month,id_inicio,co_cadena_index,...,id_month_index,id_inicio_index,co_cadena_ohe,genero_ohe,subgenero_ohe,tipo_contenido_ohe,id_weekday_ohe,id_month_ohe,id_inicio_ohe,features
0,0,1.133333,11,IF,IF,SERIE,Tuesday,April,00,128.0,...,10.0,14.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0)","(0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0)","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,0,1.483333,11,IF,IF,SERIE,Saturday,July,10,128.0,...,4.0,12.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0)","(1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)","(0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,2,16.633333,113,IF,IF,SERIE,Sunday,September,20,146.0,...,7.0,1.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0)","(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0)","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,3,1.150000,120,IF,IF,SERIE,Friday,December,18,136.0,...,0.0,5.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0)","(0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0)","(1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,4,1.983333,11,IF,MT,SERIE,Wednesday,May,08,128.0,...,8.0,15.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0)","(0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0)","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [125]:
p_model.transform(testDF).limit(5).toPandas()

,id_contenido,minutos,co_cadena,genero,subgenero,tipo_contenido,id_weekday,id_month,id_inicio,co_cadena_index,...,id_month_index,id_inicio_index,co_cadena_ohe,genero_ohe,subgenero_ohe,tipo_contenido_ohe,id_weekday_ohe,id_month_ohe,id_inicio_ohe,features
0,0,5.633333,11,IF,IF,SERIE,Saturday,October,13,128.0,...,1.0,10.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0)","(1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)","(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,5,1.350000,135,IF,IF,SERIE,Wednesday,October,16,124.0,...,1.0,8.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0)","(0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0)","(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,5,2.850000,135,IF,IF,SERIE,Monday,October,11,124.0,...,1.0,11.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0)","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0)","(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,5,9.900000,135,IF,IF,SERIE,Thursday,February,22,124.0,...,11.0,4.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0)","(0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0)","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,7,9.950000,92,IF,IF,SERIE,Monday,January,19,115.0,...,2.0,6.0,"(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 1.0, 0.0, 0.0, 0.0)","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0)","(0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [126]:
# Obtener el tamaño del vector de features
print("===========================================================================================================\n")
print("Tamaño del vector de Features\n")
print("===========================================================================================================\n")
sampleDF = p_model.transform(trainDF.limit(1))
feature_vector = sampleDF.select("features").head().features
print(f"Tamaño del vector de features: {feature_vector.size}")


Tamaño del vector de Features




[Stage 165:=================================================>   (185 + 5) / 200]

Tamaño del vector de features: 282


6. Entrenamos un árbol de regresión para predecir la variable minutos

In [127]:
from pyspark.ml.regression import DecisionTreeRegressor

# Creamos el modelo de Árbol de Regresión
dt_regressor = DecisionTreeRegressor(
    featuresCol="features",  # La columna con las características
    labelCol="minutos",      # La variable objetivo
    predictionCol="prediction",# Columna de salida con las predicciones
    maxDepth=5, #Profundidad máxima del árbol
    seed=42                  # Semilla para reproducibilidad
)

In [128]:
# Creamos la nueva Pipeline con el Árbol de Regresión
pipeline = Pipeline(stages=indexers + encoders + [vector_assembler, dt_regressor])

In [129]:
# Ajustamos el modelo con el conjunto de entrenamiento
p_model = pipeline.fit(trainDF)

In [130]:
# Realizamos las predicciones en Train y Test
train_predictions = p_model.transform(trainDF)
test_predictions = p_model.transform(testDF)

In [131]:
# Visualizamos las predicciones en train
print("====================================================================\n")
print("Predicciones en train\n")
print("====================================================================\n")

train_predictions.select(
    "minutos", "prediction", "features"
).show(5, truncate=False)


Predicciones en train


+------------------+------------------+-----------------------------------------------------------------+
|minutos           |prediction        |features                                                         |
+------------------+------------------+-----------------------------------------------------------------+
|1.1333333333333333|134.16985943015052|(282,[128,160,169,235,243,256,272],[1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
|1.4833333333333334|134.16985943015052|(282,[128,160,169,235,239,250,270],[1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
|16.633333333333333|134.16985943015052|(282,[146,160,169,235,240,253,259],[1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
|1.15              |134.16985943015052|(282,[136,160,169,235,241,246,263],[1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
|1.9833333333333334|134.16985943015052|(282,[128,160,177,235,242,254,273],[1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
+------------------+------------------+-----------------------------------------------------------------+
only showing top 5 ro

In [132]:
print("====================================================================\n")
print("Predicciones en test\n")
print("====================================================================\n")
test_predictions.select(
    "minutos", "prediction", "features"
).show(5, truncate=False)


Predicciones en test


+-----------------+------------------+-----------------------------------------------------------------+
|minutos          |prediction        |features                                                         |
+-----------------+------------------+-----------------------------------------------------------------+
|5.633333333333334|134.16985943015052|(282,[128,160,169,235,239,247,268],[1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
|1.35             |134.16985943015052|(282,[124,160,169,235,242,247,266],[1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
|2.85             |134.16985943015052|(282,[124,160,169,235,245,247,269],[1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
|9.9              |134.16985943015052|(282,[124,160,169,235,244,257,262],[1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
|9.95             |134.16985943015052|(282,[115,160,169,235,245,248,264],[1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
+-----------------+------------------+-----------------------------------------------------------------+
only showing top 5 rows



7. Evaluar el modelo resultante usando RMSE tanto en la muestra de entrenamiento como en la muestra de test: Comentar el resultado.

In [133]:

from pyspark.ml.evaluation import RegressionEvaluator

# Creamos un evaluador para RMSE
evaluator_rmse = RegressionEvaluator(
    labelCol="minutos",
    predictionCol="prediction",
    metricName="rmse"
)

# Creamos un evaluador para R²
evaluator_r2 = RegressionEvaluator(
    labelCol="minutos",
    predictionCol="prediction",
    metricName="r2"
)


In [134]:
print("====================================================================\n")
print("Evaluación del modelo\n")
print("====================================================================\n")
# RMSE en Train
rmse_train = evaluator_rmse.evaluate(train_predictions)
print(f"RMSE (Train): {rmse_train:.2f}")

# R² en Train
r2_train = evaluator_r2.evaluate(train_predictions)
print(f"R² (Train): {r2_train:.2f}")

# RMSE en Test
rmse_test = evaluator_rmse.evaluate(test_predictions)
print(f"RMSE (Test): {rmse_test:.2f}")

# R² en Test
r2_test = evaluator_r2.evaluate(test_predictions)
print(f"R² (Test): {r2_test:.2f}")


Evaluación del modelo




RMSE (Train): 672.09


R² (Train): 0.51


RMSE (Test): 719.26


[Stage 237:=================================================>   (186 + 5) / 200]

R² (Test): 0.46


**Análisis de los resultados**

En el conjunto de entrenamiento, el error medio de las predicciones es de 672.09 minutos, mientras que en el conjunto de prueba asciende a 719.26 minutos.
Estos valores indican que el modelo tiene una alta dispersión respecto a los valores reales, lo cual sugiere que las predicciones no son precisas.

El modelo explica únicamente el 51% de la variabilidad de la variable objetivo en el conjunto de entrenamiento y solo el 46% en el conjunto de prueba.
Un valor de R² cercano a 1 indica un buen ajuste, mientras que valores por debajo de, aproximadamente, 0.7, como en este caso, sugieren que el modelo no captura bien las relaciones subyacentes en los datos.

La diferencia entre las métricas en Train y Test es relativamente pequeña, lo que indica que el modelo no está sobreajustado (overfitting) ni subajustado (underfitting).

Sin embargo, el bajo rendimiento generalizado en ambos conjuntos sugiere que el Árbol de Regresión podría no ser el modelo más adecuado para este problema.

**Posibles razones del bajo rendimiento**

El Árbol de Regresión es un modelo que funciona bien cuando existen relaciones no lineales y segmentadas entre las características y la variable objetivo.
Sin embargo, si las relaciones son más complejas o dependen de interacciones sutiles entre las variables, el Árbol de Regresión puede no capturarlas adecuadamente. Así, los Árboles de Regresión tienden a fragmentar los datos en regiones rectangulares, lo que puede no adaptarse bien a problemas donde las relaciones son más sutiles.

In [135]:
print("====================================================================\n")
print("Análisis de resultados\n")
print("====================================================================\n")
print("""

En el conjunto de entrenamiento, el error medio de las predicciones es de 672.09 minutos, mientras que en el conjunto de prueba asciende a 719.26 minutos.
Estos valores indican que el modelo tiene una alta dispersión respecto a los valores reales, lo cual sugiere que las predicciones no son precisas.

El modelo explica únicamente el 51% de la variabilidad de la variable objetivo en el conjunto de entrenamiento y solo el 46% en el conjunto de prueba.
Un valor de R² cercano a 1 indica un buen ajuste, mientras que valores por debajo de, aproximadamente, 0.7, como en este caso, sugieren que el modelo no captura bien las relaciones subyacentes en los datos.

La diferencia entre las métricas en Train y Test es relativamente pequeña, lo que indica que el modelo no está sobreajustado (overfitting) ni subajustado (underfitting).

Sin embargo, el bajo rendimiento generalizado en ambos conjuntos sugiere que el Árbol de Regresión podría no ser el modelo más adecuado para este problema.

**Posibles razones del bajo rendimiento**

El Árbol de Regresión es un modelo que funciona bien cuando existen relaciones no lineales y segmentadas entre las características y la variable objetivo.
Sin embargo, si las relaciones son más complejas o dependen de interacciones sutiles entre las variables, el Árbol de Regresión puede no capturarlas adecuadamente. 
Así, los Árboles de Regresión tienden a fragmentar los datos en regiones rectangulares, lo que puede no adaptarse bien a problemas donde las relaciones son más sutiles.
""")


Análisis de resultados




En el conjunto de entrenamiento, el error medio de las predicciones es de 672.09 minutos, mientras que en el conjunto de prueba asciende a 719.26 minutos.
Estos valores indican que el modelo tiene una alta dispersión respecto a los valores reales, lo cual sugiere que las predicciones no son precisas.

El modelo explica únicamente el 51% de la variabilidad de la variable objetivo en el conjunto de entrenamiento y solo el 46% en el conjunto de prueba.
Un valor de R² cercano a 1 indica un buen ajuste, mientras que valores por debajo de, aproximadamente, 0.7, como en este caso, sugieren que el modelo no captura bien las relaciones subyacentes en los datos.

La diferencia entre las métricas en Train y Test es relativamente pequeña, lo que indica que el modelo no está sobreajustado (overfitting) ni subajustado (underfitting).

Sin embargo, el bajo rendimiento generalizado en ambos conjuntos sugiere que el Árbol de Regresión podría no ser el modelo más adecuado par

#### 13.- Modelo de Gradient_boosting 

In [136]:
# Ajustamos la pipeline en los datos de entrenamiento
print("====================================================================\n")
print("Modelo Gradient_boosting \n")
print("====================================================================\n")
print("""
**Configuración de la Cuadrícula de Hiperparámetros**

Para reducir el tiempo de ejecución durante el proceso de validación cruzada, se ha optado por restringir la búsqueda de hiperparámetros a una selección más reducida. Actualmente, la cuadrícula considera únicamente los valores de `maxDepth` [5, 7, 10]. 

Si se desea realizar una búsqueda más exhaustiva, es posible ampliar la cuadrícula descomentando las siguientes líneas en la configuración:

- `addGrid(gbt.maxIter, [100])`  # Número de iteraciones.  
- `addGrid(gbt.stepSize, [0.01, 0.1])`  # Tasa de aprendizaje.  

Sin embargo, esto incrementará significativamente el tiempo de ejecución debido al aumento exponencial en el número de combinaciones a evaluar.

Esta decisión ha sido tomada para equilibrar el tiempo de cómputo y la calidad esperada de los resultados en el contexto actual del proyecto.
""")


Modelo Gradient_boosting 



**Configuración de la Cuadrícula de Hiperparámetros**

Para reducir el tiempo de ejecución durante el proceso de validación cruzada, se ha optado por restringir la búsqueda de hiperparámetros a una selección más reducida. Actualmente, la cuadrícula considera únicamente los valores de `maxDepth` [5, 7, 10]. 

Si se desea realizar una búsqueda más exhaustiva, es posible ampliar la cuadrícula descomentando las siguientes líneas en la configuración:

- `addGrid(gbt.maxIter, [100])`  # Número de iteraciones.  
- `addGrid(gbt.stepSize, [0.01, 0.1])`  # Tasa de aprendizaje.  

Sin embargo, esto incrementará significativamente el tiempo de ejecución debido al aumento exponencial en el número de combinaciones a evaluar.

Esta decisión ha sido tomada para equilibrar el tiempo de cómputo y la calidad esperada de los resultados en el contexto actual del proyecto.



In [137]:
# Creamos el modelo Gradient Boosted Trees

from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="minutos",
    predictionCol="prediction",
    maxIter=50,      # Número máximo de iteraciones (ajustable)
    maxDepth=5,      # Profundidad máxima de los árboles (ajustable)
    seed=42
)

In [138]:
# Definimos la cuadrícula de hiperparámetros

from pyspark.ml.tuning import ParamGridBuilder

# Definir la cuadrícula de hiperparámetros usando paréntesis
paramGrid = (ParamGridBuilder()
    .addGrid(gbt.maxDepth, [5,7,10])  # Usar la lista dinámica
    #.addGrid(gbt.maxIter, [100])  # Iteraciones
    #.addGrid(gbt.stepSize, [0.01, 0.1])  # Tasa de aprendizaje
    .build()
)

In [139]:
# Creamos el evaluador
evaluator = RegressionEvaluator(
    labelCol="minutos",
    predictionCol="prediction",
    metricName="rmse"
)

In [140]:
from pyspark.ml.tuning import CrossValidator

# Crear el CrossValidator
crossval = CrossValidator(
    estimator=gbt,           # Modelo a ajustar
    estimatorParamMaps=paramGrid,  # Cuadrícula de hiperparámetros
    evaluator=evaluator,     # Métrica de evaluación
    numFolds=5,              # Número de folds para la validación cruzada
    seed=42
)

In [141]:
# Creamos la Pipeline
pipeline = Pipeline(stages=indexers + encoders + [vector_assembler, crossval])

In [142]:
# Ajustamos la pipeline en los datos de entrenamiento
print("====================================================================\n")
print("Modelo Gradient_boosting \n")
print("====================================================================\n")
print("Entrenando modelo ... \n")


Modelo Gradient_boosting 


Entrenando modelo ... 



In [113]:
cv_model = pipeline.fit(trainDF)

                                                                                25/01/08 01:14:31 ERROR ContextCleaner: Error cleaning broadcast 516
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.storage.BlockManagerMaster.removeBroadcast(BlockManagerMaster.scala:222)
	at org.apache.spark.broadcast.TorrentBroadcast$.unpersist(TorrentBroadcast.scala:395)
	at org.apache.spark.broadcast.TorrentBroadcastFactory.unbroadcast(TorrentBroadcastFactory.scala:49)
	at org.apache.spark.broadcast.BroadcastManager.unbroadcast(BroadcastManager.scala:82)
	at org.apache.spark.ContextCleaner.doCleanupBroadcast(ContextCleaner.scala:256)
	at org.apache.spark.ContextCleaner.$anonfun$keepCleaning$3(ContextCleaner.scala:204)
	at org.apache.spark.C

In [114]:
# Guardar el modelo entrenado en el directorio actual
cv_model.save("./gbt_modelo")

In [115]:
# Obtenemos el mejor modelo
best_model = cv_model.stages[-1].bestModel
print("====================================================================\n")
print("Mejores hiperparámetros\n")
print("====================================================================\n")
# Mostramos los hiperparámetros óptimos
print(f" Mejor maxDepth: {best_model.getOrDefault('maxDepth')}")
print(f" Mejor maxIter: {best_model.getOrDefault('maxIter')}")
print(f" Mejor stepSize: {best_model.getOrDefault('stepSize')}")

 Mejor maxDepth: 10
 Mejor maxIter: 50
 Mejor stepSize: 0.1


In [116]:
print("====================================================================\n")
print("Evaluación del modelo\n")
print("====================================================================\n")
# Calculamos las Predicciones en Train y Test
train_predictions = cv_model.transform(trainDF)
test_predictions = cv_model.transform(testDF)

# Evaluamos RMSE y R² en Train
rmse_train = evaluator.evaluate(train_predictions)
evaluator.setMetricName("r2")
r2_train = evaluator.evaluate(train_predictions)

# Evaluamos RMSE y R² en Test
evaluator.setMetricName("rmse")
rmse_test = evaluator.evaluate(test_predictions)
evaluator.setMetricName("r2")
r2_test = evaluator.evaluate(test_predictions)

# Mostramos resultados
print(f" RMSE (Train): {rmse_train:.2f}")
print(f" R² (Train): {r2_train:.2f}")
print(f" RMSE (Test): {rmse_test:.2f}")
print(f" R² (Test): {r2_test:.2f}")

[Stage 18472:===============================================>  (190 + 10) / 200]

 RMSE (Train): 387.86
 R² (Train): 0.84
 RMSE (Test): 506.55
 R² (Test): 0.73


In [143]:
spark.stop()